In [ ]:
import numpy as np
from cellpose import models, core, io
from spotiflow.model import Spotiflow
from pathlib import Path
import os
import apoc
from tqdm import tqdm
import pyclesperanto_prototype as cle
from utils import list_images, read_image, brightfield_correction, detect_infected_cells, filter_points_interpolated

io.logger_setup()  # run this to get printing of progress

# Check if notebook has GPU access
if core.use_gpu() == False:
    raise ImportError("No GPU access, change your runtime")

# Activate .cle GPU acceleration
cle.select_device("RTX")

# Load pre-trained Cellpose-SAM and Spotiflow models
model = models.CellposeModel(gpu=True)
spotiflow_model = Spotiflow.from_pretrained("general")

In [ ]:
# Load pretrained Object Classifier for Mycobacterium infection detection
mtb_cl_filename = "./pretrained_classifiers/no_nuclei_signal/siMtb screen I_LØ/Mtb_segmenter.cl"
mtb_segmenter = apoc.ObjectSegmenter(opencl_filename=mtb_cl_filename)

# Same folder list as batch notebook
main_directory_path = Path("X:\Lisa\siMtb screen I_LØ")

folders_with_nuc = [
    name for name in os.listdir(main_directory_path)
    if os.path.isdir(os.path.join(main_directory_path, name)) and "Nuc" not in name and "Results" not in name
]

# Pick one folder and one image for visualization
folder = folders_with_nuc[0]
directory_path = main_directory_path / folder
images = list_images(directory_path)
image = images[0]

print(f"Folder: {folder}")
print(f"Image: {image}")

# Puncta markers (same as batch notebook)
puncta_markers = [("LC3B", 0), ("GAL3", 1), ("Chmp4B", 2)]

In [ ]:
slicing_factor_xy = None  # Use 2 or 4 for downsampling in xy (None for lossless)

# Brightfield correction (uses cached bf_correction.tiff if present)
bf_correction = brightfield_correction(directory_path, images, slicing_factor_xy)

# Read single image
img, filename = read_image(image, slicing_factor_xy)

plate_nr = filename.split("_")[0]
well_id = filename.split("Wells-")[1].split("__")[0]
print(f"Plate: {plate_nr}, Well: {well_id}")

In [ ]:
# Predict cytoplasm labels using CellposeSAM
# LC3B(img[0]) and GAL3(img[1]) are combined to create the "cytoplasm" channel, a marker agnostic input is the corrected brightfield image
# CellposeSAM is invariant to channel ordering 
cell_labels, flows, styles = model.eval(
    np.stack((img[[0, 1]].sum(axis=0), (img[4] - bf_correction)), axis=0
), niter=1000)

# Infection detection (same as batch)
infection_stats = []
infected_labels = detect_infected_cells(
    img, mtb_segmenter, cell_labels, plate_nr, well_id, infection_stats
)

# Mtb labels for Napari (detect_infected_cells does not return mtb_labels)
mtb_labels = mtb_segmenter.predict(img[3])
mtb_labels = cle.pull(mtb_labels)

# Infected-only cytoplasm for optional labels layer
infected_cytoplasm = np.where(
    np.isin(cell_labels, infected_labels), cell_labels, 0
).astype(cell_labels.dtype)

In [ ]:
# Puncta detection: same logic as utils.puncta_detection, keep points for Napari
puncta_points = {}
for puncta_marker in puncta_markers:
    name, ch_nr = puncta_marker[0], puncta_marker[1]
    puncta_cl_filename = f"./pretrained_classifiers/no_nuclei_signal/siMtb screen I_LØ/{name}_segmenter.cl"
    puncta_segmenter = apoc.PixelClassifier(opencl_filename=puncta_cl_filename)
    points, details = spotiflow_model.predict(img[ch_nr], subpix=True, verbose=False)
    mask = puncta_segmenter.predict(img[ch_nr])
    filtered_points = filter_points_interpolated(points, mask)
    puncta_points[name] = filtered_points
    print(f"{name}: {len(filtered_points)} spots")

print("Puncta points ready for Napari.")

In [ ]:
import napari

viewer = napari.Viewer(ndisplay=2)

# Image layers: input channels used for segmentation and key signals
viewer.add_image(img, name="input_image")
viewer.add_image(img[4] - bf_correction, name="BF_corrected")
viewer.add_image(img[[0, 1]].sum(axis=0), name="cytoplasm_fluo_sum")
viewer.add_image(img[3], name="Mtb_ch")

# Labels
viewer.add_labels(cell_labels, name="cell_labels")
viewer.add_labels(mtb_labels, name="Mtb_labels")
viewer.add_labels(infected_cytoplasm, name="infected_cell_labels")

# Puncta points (distinct colors per marker)
colors = {"LC3B": "red", "GAL3": "yellow", "Chmp4B": "green"}
for name, pts in puncta_points.items():
    if len(pts) > 0:
        viewer.add_points(pts, face_color=colors.get(name, "white"), name=f"puncta_{name}")
    else:
        viewer.add_points(np.zeros((0, 2)), face_color=colors.get(name, "white"), name=f"puncta_{name}")